# Notebook 1: Data Curation, Standardization and Database Ingestion (Person A)

**Output:** `pmos_knowledge_base.db`, a deduplicated, provenance-tracked SQLite database that Persons B and C build on.

**Pipeline:** Data Collection → Cleaning → Identifier Mapping → Validation, as defined in the progress document.

| Step | What it does |
|---|---|
| 0 | Setup, folders, config, contract with B and C |
| 1 | Seed list, checked against Open Targets PCOS associations |
| 2 | Harvest all seven sources through their APIs; save every raw response |
| 3 | Clean into one DataFrame per entity type |
| 4 | Identifier mapping and cross-source check |
| 5 | Revised schema |
| 6 | Idempotent upsert ingestion |
| 7 | Validation assertions |

**Run it top to bottom.** The first run makes a few hundred API calls (several minutes). Every response is cached in `data/raw/`, so later runs are fast and work offline. Delete a raw file to force a refresh.

**ID convention:** IDs are stored bare (`1586`, `4091`). The namespace lives in the column name (`ncbi_gene_id`, `pubchem_cid`, `uniprot_accession`, `reactome_id`).

## Step 0: Setup and contract

In [1]:
import json, re, time, hashlib, sqlite3, random
import datetime as dt
from pathlib import Path
from urllib.parse import quote
import pandas as pd
import requests

print("[Step 0/8] Initializing environment and directory paths...")
ROOT = Path(".")
RAW, CLEAN = ROOT / "data" / "raw", ROOT / "data" / "clean"
for d in (RAW, CLEAN):
    d.mkdir(parents=True, exist_ok=True)
print(f"  -> Created/verified directories: {RAW} and {CLEAN}")

DB_PATH = ROOT / "pmos_knowledge_base.db"
SEED_CSV = ROOT / "seed_entities.csv"

# ---- config: edit these ----
CONTACT_EMAIL = "saarikarajen@gmail.com"   
NCBI_API_KEY = None                        
TAXON = 9606                               
PCOS_EFO = "MONDO_0008487"                   
PUBS_PER_ENTITY = 3                        
REBUILD_DB = True                          

CONTRACT_KEYS = ["gene_symbol", "protein_id", "pathway_name", "drug_name", "pmid"]
print(f"  -> Contract keys defined: {CONTRACT_KEYS}")

unresolved = []   

def log_unresolved(entity_type, name, reason):
    unresolved.append({"entity_type": entity_type, "name": name, "reason": reason})
    print(f"  [unresolved] {entity_type}: {name} -> {reason}")

session = requests.Session()
session.headers.update({"User-Agent": f"pmos-kb/0.1 ({CONTACT_EMAIL})"})
print("[Step 0/8] Setup and configuration complete successfully.\n")

# ---- HTTP helper: every call is cached to data/raw/ with source, query and timestamp ----
session = requests.Session()
session.headers.update({"User-Agent": f"pmos-kb/0.1 ({CONTACT_EMAIL})"})

def _slug(s):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(s)).strip("_")[:50]

def fetch(source, query, url, params=None, json_body=None, delay=0.4):
    """GET (or POST when json_body is given). Returns the saved wrapper:
    {source, query, url, params, retrieved_at, response}. A 404 is stored as response=None."""
    key = hashlib.sha1(json.dumps([url, params, json_body], sort_keys=True, default=str).encode()).hexdigest()[:10]
    path = RAW / f"{source}__{_slug(query)}__{key}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    resp = None
    for attempt in range(4):
        try:
            if json_body is not None:
                r = session.post(url, json=json_body, timeout=60)
            else:
                r = session.get(url, params=params, timeout=60)
            if r.status_code == 404:
                break
            r.raise_for_status()
            resp = r.json()
            break
        except (requests.RequestException, ValueError):
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)
    record = {"source": source, "query": str(query), "url": url, "params": params,
              "retrieved_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
              "response": resp}
    path.write_text(json.dumps(record, indent=1, ensure_ascii=False), encoding="utf-8")
    time.sleep(delay)
    return record

def day(record):
    return record["retrieved_at"][:10]

[Step 0/8] Initializing environment and directory paths...
  -> Created/verified directories: data\raw and data\clean
  -> Contract keys defined: ['gene_symbol', 'protein_id', 'pathway_name', 'drug_name', 'pmid']
[Step 0/8] Setup and configuration complete successfully.



In [2]:
(ROOT / "README.md").write_text("""# PMOS knowledge base: data layer (Person A)

## Contract with Persons B and C
Fixed column names (do not rename): gene_symbol, protein_id, pathway_name, drug_name, pmid.
Added columns are allowed. All tables carry provenance: source, source_accession, retrieved_on.
Foreign keys are enforced only if every connection runs `PRAGMA foreign_keys = ON;`.

## Layout
- notebook1.ipynb          rerunnable pipeline
- seed_entities.csv        entities to harvest
- data/raw/                every API response, with source, query and timestamp
- data/clean/              cleaned CSVs that feed the database
- pmos_knowledge_base.db   output

## ID convention
IDs are stored bare (1586, 4091); the namespace is in the column name.

## Data dictionary
Generated at the end of Step 7 (see below this line once the notebook has run).
""", encoding="utf-8")
print("Setup complete:", RAW, CLEAN)

Setup complete: data\raw data\clean


## Step 1: Seed list

Columns: `entity_type, name, notes` (as planned) plus two helper columns:
- `lookup`: the term sent to the APIs (gene symbol, compound name, search phrase)
- `id_source`: which database supplies the canonical ID for hormones and biomarkers (`pubchem` for small molecules, `uniprot` for protein hormones)

Tier 1 below is deliberately small. Get it working end to end, then add rows. The `215 / 18 / 52 / 31` counts in the UI mockup are placeholders, not targets.

In [3]:
SEED_COLUMNS = ["entity_type", "name", "lookup", "id_source", "notes"]
seed_rows = [
    # genes (tier 1)
    *[("gene", g, g, "", n) for g, n in [
        ("CYP17A1", "from progress doc; steroidogenesis"), ("CYP19A1", "aromatase"), ("CYP11A1", "steroidogenesis"),
        ("AR", "androgen receptor"), ("INSR", "insulin receptor"), ("LHCGR", "LH receptor"),
        ("FSHR", "FSH receptor"), ("AMH", "reproductive biomarker gene"), ("AMHR2", "AMH receptor"),
        ("SHBG", "binds sex steroids")]],
    # hormones
    ("hormone", "Testosterone", "testosterone", "pubchem", "from progress doc"),
    ("hormone", "Estradiol", "estradiol", "pubchem", ""),
    ("hormone", "Androstenedione", "androstenedione", "pubchem", ""),
    ("hormone", "Progesterone", "progesterone", "pubchem", ""),
    ("hormone", "Luteinizing hormone", "LHB", "uniprot", "protein hormone: no PubChem compound; beta chain"),
    ("hormone", "Follicle-stimulating hormone", "FSHB", "uniprot", "protein hormone; beta chain"),
    ("hormone", "Insulin", "INS", "uniprot", "protein hormone"),
    # biomarkers (ASCII name avoids Mullerian/Muellerian spelling splits)
    ("biomarker", "Anti-Mullerian hormone", "AMH", "uniprot", "from progress doc"),
    ("biomarker", "Sex hormone-binding globulin", "SHBG", "uniprot", ""),
    ("biomarker", "INSL3", "INSL3", "uniprot", "shown in UI mockup"),
    # therapeutics
    ("therapeutic", "Metformin", "metformin", "pubchem", "from progress doc"),
    ("therapeutic", "Spironolactone", "spironolactone", "pubchem", ""),
    ("therapeutic", "Clomiphene", "clomiphene", "pubchem", ""),
    ("therapeutic", "Letrozole", "letrozole", "pubchem", ""),
    ("therapeutic", "Pioglitazone", "pioglitazone", "pubchem", ""),
    # pathways: KEGG-style name from the doc; harvest searches Reactome and records Reactome's own name and ID
    ("pathway", "Steroid hormone biosynthesis", "steroid hormone biosynthesis", "reactome",
     "KEGG-style name from progress doc; the Reactome match is logged for review"),
]
if SEED_CSV.exists():
    print("seed_entities.csv exists: keeping your edited version")
else:
    pd.DataFrame(seed_rows, columns=SEED_COLUMNS).to_csv(SEED_CSV, index=False)
    print("seed_entities.csv created")
seed = pd.read_csv(SEED_CSV, dtype=str).fillna("")
seed.groupby("entity_type").size()

seed_entities.csv exists: keeping your edited version


entity_type
biomarker       7
gene           36
hormone        17
pathway         6
therapeutic    13
dtype: int64

### Evidence check against Open Targets

Pulls the PCOS gene associations and compares your seed genes with them. Genes in the seed list but absent from the top associations are not wrong, but should be justified. The second table lists strongly associated genes you have not seeded yet.

In [4]:
OT_URL = "https://api.platform.opentargets.org/api/v4/graphql"
OT_QUERY = """
query pcos($efo: String!, $index: Int!, $size: Int!) {
  disease(efoId: $efo) {
    id name
    associatedTargets(page: {index: $index, size: $size}) {
      count
      rows { target { id approvedSymbol } score }
    }
  }
}"""

def open_targets_pcos(max_pages=5, size=100):
    rows, name, retrieved = [], None, None
    for page in range(max_pages):
        rec = fetch("opentargets", f"{PCOS_EFO}_p{page}", OT_URL,
                    json_body={"query": OT_QUERY, "variables": {"efo": PCOS_EFO, "index": page, "size": size}})
        d = ((rec["response"] or {}).get("data") or {}).get("disease")
        if not d:
            break
        name, retrieved = d["name"], day(rec)
        batch = d["associatedTargets"]["rows"]
        rows += [{"gene_symbol": r["target"]["approvedSymbol"].upper(), "ensembl_id": r["target"]["id"],
                  "ot_score": r["score"]} for r in batch]
        if len(batch) < size:
            break
    assert name and "polycystic" in name.lower(), f"{PCOS_EFO} is not PCOS (got {name!r}); fix PCOS_EFO"
    df = pd.DataFrame(rows).drop_duplicates("gene_symbol").sort_values("ot_score", ascending=False)
    df["ot_rank"] = range(1, len(df) + 1)
    df["retrieved_on"] = retrieved
    return df.reset_index(drop=True)

ot = open_targets_pcos()
ot.to_csv(CLEAN / "open_targets_pcos.csv", index=False)
ot_score = dict(zip(ot.gene_symbol, ot.ot_score))
seed_genes = seed.loc[seed.entity_type == "gene", "lookup"].str.upper()
print(f"{len(ot)} PCOS-associated targets retrieved\n")
print("Seed genes vs Open Targets:")
print(ot[ot.gene_symbol.isin(seed_genes)][["gene_symbol", "ot_rank", "ot_score"]].to_string(index=False))
print("\nSeed genes NOT in Open Targets PCOS list:", sorted(set(seed_genes) - set(ot.gene_symbol)) or "none")
print("\nTop associated genes not yet seeded (candidates for tier 2):")
print(ot[~ot.gene_symbol.isin(seed_genes)].head(15)[["gene_symbol", "ot_rank", "ot_score"]].to_string(index=False))

500 PCOS-associated targets retrieved

Seed genes vs Open Targets:
gene_symbol  ot_rank  ot_score
       ESR1        1  0.600350
      ERBB4        2  0.554508
      GNRHR        3  0.537614
        FTO        4  0.522636
      CHEK2        5  0.496632
       FSHB        6  0.495947
      AOPEP        7  0.491651
    DENND1A        8  0.481205
       FSHR        9  0.474815
      GNAI1       10  0.469898
      PROX1       13  0.451220
     ZBTB16       14  0.441049
    CYP19A1       15  0.439559
       IRF1       16  0.437185
       SUOX       17  0.434401
      LHCGR       18  0.432337
      GLP1R       21  0.422301
        PGR       22  0.421427
      GATA4       24  0.421333
       YAP1       73  0.418626
      THADA       77  0.383737
      PPARG       78  0.377992
      HMGA2       82  0.348019
       LEPR       95  0.292722
       INSR      103  0.265228
         AR      155  0.252103
       TOX3      178  0.213108
      RAB5B      196  0.170246
        AMH      206  0.132739
   

## Step 2: Harvest programmatically

One function per source. Each returns plain dicts with `source`, `source_accession` and `retrieved_on`, and every API response is already on disk in `data/raw/` via `fetch()`. Human only; reviewed (Swiss-Prot) UniProt entries only.

In [5]:
# ---------------- NCBI Gene & PubMed (E-utilities) ----------------
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"

def eutils(endpoint, source, query, **params):
    params.update({"retmode": "json", "tool": "pmos-kb", "email": CONTACT_EMAIL})
    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY
    return fetch(source, query, EUTILS + endpoint, params, delay=0.12 if NCBI_API_KEY else 0.4)

def ncbi_gene(symbol):
    s = eutils("esearch.fcgi", "ncbigene", symbol, db="gene", term=f"{symbol}[sym] AND {TAXON}[taxid]")
    ids = ((s["response"] or {}).get("esearchresult") or {}).get("idlist", [])
    if not ids:
        return None
    gid = ids[0]
    d = eutils("esummary.fcgi", "ncbigene", symbol, db="gene", id=gid)
    doc = ((d["response"] or {}).get("result") or {}).get(gid)
    if not doc:
        return None
    return {"gene_symbol": doc.get("name"), "name": doc.get("description"), "ncbi_gene_id": gid,
            "description": doc.get("summary") or doc.get("description"),
            "source": "NCBI Gene", "source_accession": gid, "retrieved_on": day(d)}

def pubmed_for(term, n=PUBS_PER_ENTITY):
    q = f"{term} AND polycystic ovary syndrome"
    s = eutils("esearch.fcgi", "pubmed_search", term, db="pubmed", term=q, retmax=n, sort="relevance")
    ids = ((s["response"] or {}).get("esearchresult") or {}).get("idlist", [])
    if not ids:
        return []
    d = eutils("esummary.fcgi", "pubmed_summary", term, db="pubmed", id=",".join(ids))
    result = (d["response"] or {}).get("result") or {}
    out = []
    for pmid in ids:
        doc = result.get(pmid)
        if not doc:
            continue
        m = re.search(r"\d{4}", doc.get("pubdate", "") or "")
        out.append({"pmid": pmid, "title": doc.get("title"), "year": m.group(0) if m else None,
                    "journal": doc.get("source"), "source": "PubMed", "source_accession": pmid,
                    "retrieved_on": day(d)})
    return out

# ---------------- UniProt ----------------
def uniprot(symbol):
    rec = fetch("uniprot", symbol, "https://rest.uniprot.org/uniprotkb/search", {
        "query": f"(gene_exact:{symbol}) AND (organism_id:{TAXON}) AND (reviewed:true)",
        "format": "json", "size": 5,
        "fields": "accession,protein_name,gene_primary,cc_function,xref_geneid"})
    results = (rec["response"] or {}).get("results") or []
    if not results:
        return None
    r = next((x for x in results
               if (x.get("genes") or [{}])[0].get("geneName", {}).get("value", "").upper() == symbol.upper()),
              results[0])
    name = (((r.get("proteinDescription") or {}).get("recommendedName") or {}).get("fullName") or {}).get("value")
    func = next((t["value"] for c in r.get("comments", []) if c.get("commentType") == "FUNCTION"
                 for t in c.get("texts", [])), None)
    geneid = next((x["id"] for x in r.get("uniProtKBCrossReferences", []) if x.get("database") == "GeneID"), None)
    return {"protein_id": r["primaryAccession"], "name": name, "function": func,
            "gene_symbol": symbol.upper(), "uniprot_geneid": geneid,
            "source": "UniProt", "source_accession": r["primaryAccession"], "retrieved_on": day(rec)}

# ---------------- Reactome ----------------
REACTOME = "https://reactome.org/ContentService/"

def _strip_html(s):
    return re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", s or "")).strip() or None

def reactome_detail(st_id):
    rec = fetch("reactome", st_id, REACTOME + f"data/query/{st_id}")
    r = rec["response"] or {}
    summ = r.get("summation") or []
    return {"pathway_name": r.get("displayName"), "reactome_id": r.get("stId", st_id),
            "description": _strip_html(summ[0].get("text")) if summ else None,
            "source": "Reactome", "source_accession": r.get("stId", st_id), "retrieved_on": day(rec)}

def reactome_for_uniprot(acc):
    rec = fetch("reactome", f"map_{acc}", REACTOME + f"data/mapping/UniProt/{acc}/pathways", {"species": TAXON})
    return [p["stId"] for p in (rec["response"] or []) if isinstance(p, dict) and p.get("stId")]

def reactome_search(term):
    rec = fetch("reactome", f"search_{term}", REACTOME + "search/query",
                {"query": term, "species": "Homo sapiens", "types": "Pathway", "cluster": "true"})
    hits = []
    for grp in (rec["response"] or {}).get("results", []):
        hits += [e for e in grp.get("entries", []) if e.get("stId")]
    return hits

# ---------------- Gene Ontology (QuickGO) ----------------
QUICKGO = "https://www.ebi.ac.uk/QuickGO/services/"

def quickgo(acc):
    rec = fetch("quickgo", acc, QUICKGO + "annotation/search", {"geneProductId": acc, "limit": 200})
    anns = (rec["response"] or {}).get("results") or []
    ids = sorted({a["goId"] for a in anns if a.get("goId")})
    names = {}
    for i in range(0, len(ids), 50):
        chunk = ids[i:i + 50]
        t = fetch("quickgo_terms", f"{acc}_{i}", QUICKGO + "ontology/go/terms/" + ",".join(chunk))
        for r in (t["response"] or {}).get("results", []):
            names[r["id"]] = (r.get("name"), r.get("aspect"))
    out = []
    for go_id in ids:
        nm, aspect = names.get(go_id, (None, None))
        if aspect in ("biological_process", "molecular_function"):
            out.append({"protein_id": acc, "go_id": go_id, "go_name": nm, "aspect": aspect,
                        "source": "Gene Ontology (QuickGO)", "source_accession": go_id, "retrieved_on": day(rec)})
    return out

# ---------------- PubChem ----------------
PUG = "https://pubchem.ncbi.nlm.nih.gov/rest/"

def _strings(node):
    if isinstance(node, dict):
        if isinstance(node.get("String"), str):
            yield node["String"]
        for v in node.values():
            yield from _strings(v)
    elif isinstance(node, list):
        for v in node:
            yield from _strings(v)

def pubchem(name, want_mechanism=False):
    rec = fetch("pubchem", name, PUG + f"pug/compound/name/{quote(name)}/cids/JSON")
    cids = ((rec["response"] or {}).get("IdentifierList") or {}).get("CID") or []
    if not cids:
        return None
    cid = str(cids[0])
    d = fetch("pubchem_desc", cid, PUG + f"pug/compound/cid/{cid}/description/JSON")
    infos = ((d["response"] or {}).get("InformationList") or {}).get("Information") or []
    desc = next((i["Description"] for i in infos if i.get("Description")), None)
    mech = None
    if want_mechanism:
        m = fetch("pubchem_mech", cid, PUG + f"pug_view/data/compound/{cid}/JSON", {"heading": "Mechanism of Action"})
        texts = list(dict.fromkeys(_strings(m["response"] or {})))
        mech = " ".join(texts[:2])[:1000] or None
    return {"pubchem_cid": cid, "description": desc, "mechanism": mech or desc,
            "source": "PubChem", "source_accession": cid, "retrieved_on": day(rec)}

### Harvest loop

Walks the seed list, calls the functions above and collects raw records. Anything that cannot be resolved is logged, not dropped silently.

In [6]:
raw = {k: [] for k in ["genes", "proteins", "pathways", "protein_pathway", "go", "hormones",
                       "biomarkers", "therapeutics", "publications", "entity_pubs"]}
pathway_ids_seen = set()

def add_pathway(st_id):
    if st_id not in pathway_ids_seen:
        pathway_ids_seen.add(st_id)
        raw["pathways"].append(reactome_detail(st_id))

for _, row in seed.iterrows():
    et, name, lookup, id_src = row.entity_type, row["name"], row["lookup"], row.id_source
    print(f"{et}: {name}")
    print(f"[{_/len(seed)*100:.1f}% done] Harvesting {et.upper()}: '{name}' (lookup: '{lookup}') using source spec: {id_src or 'default'}")

    if et == "gene":
        g = ncbi_gene(lookup)
        if not g:
            log_unresolved("gene", name, "not found in NCBI Gene (human)")
            continue
        if g["gene_symbol"].upper() != lookup.upper():
            log_unresolved("gene", name, f"NCBI returned symbol {g['gene_symbol']}")
        raw["genes"].append(g)
        p = uniprot(lookup)
        if not p:
            log_unresolved("gene", name, "no reviewed UniProt entry")
            continue
        raw["proteins"].append(p)
        for st_id in reactome_for_uniprot(p["protein_id"]):
            raw["protein_pathway"].append({"protein_id": p["protein_id"], "gene_symbol": lookup.upper(), "reactome_id": st_id})
            add_pathway(st_id)
        raw["go"] += quickgo(p["protein_id"])

    elif et in ("hormone", "biomarker"):
        entry = {"retrieved_on": None}
        if id_src == "pubchem":
            c = pubchem(lookup)
            if not c:
                log_unresolved(et, name, "no PubChem compound"); continue
            entry = {"pubchem_cid": c["pubchem_cid"], "uniprot_accession": None, "biological_role": c["description"],
                     "description": c["description"], "source": c["source"],
                     "source_accession": c["source_accession"], "retrieved_on": c["retrieved_on"]}
        else:
            p = uniprot(lookup)
            if not p:
                log_unresolved(et, name, "no reviewed UniProt entry"); continue
            entry = {"pubchem_cid": None, "uniprot_accession": p["protein_id"], "biological_role": p["function"],
                     "description": p["function"], "source": p["source"],
                     "source_accession": p["source_accession"], "retrieved_on": p["retrieved_on"]}
        if et == "hormone":
            raw["hormones"].append({"hormone_name": name, **entry})
        else:
            score = ot_score.get(lookup.upper())
            ev = f"Open Targets PCOS association score {score:.3f}" if score is not None else None
            if ev is None:
                log_unresolved("biomarker", name, "no Open Targets PCOS association found (evidence left NULL)")
            raw["biomarkers"].append({"biomarker_name": name, "evidence": ev, **entry})

    elif et == "therapeutic":
        c = pubchem(lookup, want_mechanism=True)
        if not c:
            log_unresolved("therapeutic", name, "no PubChem compound"); continue
        raw["therapeutics"].append({"drug_name": name, **{k: c[k] for k in
                                    ("mechanism", "pubchem_cid", "source", "source_accession", "retrieved_on")}})

    elif et == "pathway":
        hits = reactome_search(lookup)
        if not hits:
            log_unresolved("pathway", name, "no Reactome match"); continue
        print("    Reactome candidates (review!):")
        for h in hits[:5]:
            print(f"      {h['stId']}  {_strip_html(h.get('name'))}")
        add_pathway(hits[0]["stId"])

    # supporting publications (not for pathways)
    if et != "pathway":
        for pub in pubmed_for(lookup if et in ("gene", "therapeutic") else name):
            raw["publications"].append(pub)
            raw["entity_pubs"].append({"entity_type": et, "entity_name": name, "pmid": pub["pmid"]})

print("\nHarvest counts:", {k: len(v) for k, v in raw.items()})

gene: CYP17A1
[0.0% done] Harvesting GENE: 'CYP17A1' (lookup: 'CYP17A1') using source spec: default
gene: CYP19A1
[1.3% done] Harvesting GENE: 'CYP19A1' (lookup: 'CYP19A1') using source spec: default
gene: CYP11A1
[2.5% done] Harvesting GENE: 'CYP11A1' (lookup: 'CYP11A1') using source spec: default
gene: AR
[3.8% done] Harvesting GENE: 'AR' (lookup: 'AR') using source spec: default
gene: INSR
[5.1% done] Harvesting GENE: 'INSR' (lookup: 'INSR') using source spec: default
gene: LHCGR
[6.3% done] Harvesting GENE: 'LHCGR' (lookup: 'LHCGR') using source spec: default
gene: FSHR
[7.6% done] Harvesting GENE: 'FSHR' (lookup: 'FSHR') using source spec: default
gene: AMH
[8.9% done] Harvesting GENE: 'AMH' (lookup: 'AMH') using source spec: default
gene: AMHR2
[10.1% done] Harvesting GENE: 'AMHR2' (lookup: 'AMHR2') using source spec: default
gene: SHBG
[11.4% done] Harvesting GENE: 'SHBG' (lookup: 'SHBG') using source spec: default
hormone: Testosterone
[12.7% done] Harvesting HORMONE: 'Testoste

gene: GNRHR
[35.4% done] Harvesting GENE: 'GNRHR' (lookup: 'GNRHR') using source spec: default
gene: FTO
[36.7% done] Harvesting GENE: 'FTO' (lookup: 'FTO') using source spec: default
gene: CHEK2
[38.0% done] Harvesting GENE: 'CHEK2' (lookup: 'CHEK2') using source spec: default


gene: FSHB
[39.2% done] Harvesting GENE: 'FSHB' (lookup: 'FSHB') using source spec: default
gene: AOPEP
[40.5% done] Harvesting GENE: 'AOPEP' (lookup: 'AOPEP') using source spec: default
gene: DENND1A
[41.8% done] Harvesting GENE: 'DENND1A' (lookup: 'DENND1A') using source spec: default
gene: GNAI1
[43.0% done] Harvesting GENE: 'GNAI1' (lookup: 'GNAI1') using source spec: default
gene: PROX1
[44.3% done] Harvesting GENE: 'PROX1' (lookup: 'PROX1') using source spec: default
gene: ZBTB16
[45.6% done] Harvesting GENE: 'ZBTB16' (lookup: 'ZBTB16') using source spec: default


gene: IRF1
[46.8% done] Harvesting GENE: 'IRF1' (lookup: 'IRF1') using source spec: default
gene: SUOX
[48.1% done] Harvesting GENE: 'SUOX' (lookup: 'SUOX') using source spec: default
gene: GLP1R
[49.4% done] Harvesting GENE: 'GLP1R' (lookup: 'GLP1R') using source spec: default
gene: PGR
[50.6% done] Harvesting GENE: 'PGR' (lookup: 'PGR') using source spec: default
gene: GATA4
[51.9% done] Harvesting GENE: 'GATA4' (lookup: 'GATA4') using source spec: default
gene: THADA
[53.2% done] Harvesting GENE: 'THADA' (lookup: 'THADA') using source spec: default
gene: YAP1
[54.4% done] Harvesting GENE: 'YAP1' (lookup: 'YAP1') using source spec: default
gene: RAB5B
[55.7% done] Harvesting GENE: 'RAB5B' (lookup: 'RAB5B') using source spec: default
gene: HMGA2
[57.0% done] Harvesting GENE: 'HMGA2' (lookup: 'HMGA2') using source spec: default
gene: TOX3
[58.2% done] Harvesting GENE: 'TOX3' (lookup: 'TOX3') using source spec: default
gene: IRS1
[59.5% done] Harvesting GENE: 'IRS1' (lookup: 'IRS1') usi

gene: TCF7L2
[64.6% done] Harvesting GENE: 'TCF7L2' (lookup: 'TCF7L2') using source spec: default
hormone: Dehydroepiandrosterone
[65.8% done] Harvesting HORMONE: 'Dehydroepiandrosterone' (lookup: 'dehydroepiandrosterone') using source spec: pubchem
hormone: Dehydroepiandrosterone sulfate
[67.1% done] Harvesting HORMONE: 'Dehydroepiandrosterone sulfate' (lookup: 'dehydroepiandrosterone sulfate') using source spec: pubchem
hormone: 17-Hydroxyprogesterone
[68.4% done] Harvesting HORMONE: '17-Hydroxyprogesterone' (lookup: '17-hydroxyprogesterone') using source spec: pubchem
hormone: Dihydrotestosterone
[69.6% done] Harvesting HORMONE: 'Dihydrotestosterone' (lookup: 'dihydrotestosterone') using source spec: pubchem
hormone: Cortisol
[70.9% done] Harvesting HORMONE: 'Cortisol' (lookup: 'cortisol') using source spec: pubchem
hormone: Prolactin
[72.2% done] Harvesting HORMONE: 'Prolactin' (lookup: 'PRL') using source spec: uniprot
hormone: Leptin
[73.4% done] Harvesting HORMONE: 'Leptin' (loo

## Step 3: Clean

One function per entity type, each returning a DataFrame with the final column names:
trim whitespace, empty strings to NULL, normalise keys, drop duplicates, parse years as integers. Output goes to `data/clean/*.csv`.

In [7]:
PROV = ["source", "source_accession", "retrieved_on"]
COLS = {
    "genes": ["gene_symbol", "name", "ncbi_gene_id", "uniprot_accession", "description"] + PROV,
    "proteins": ["protein_id", "name", "function", "gene_symbol"] + PROV,
    "hormones": ["hormone_name", "biological_role", "pubchem_cid", "uniprot_accession"] + PROV,
    "biomarkers": ["biomarker_name", "evidence", "description", "uniprot_accession"] + PROV,
    "pathways": ["pathway_name", "reactome_id", "description"] + PROV,
    "therapeutics": ["drug_name", "mechanism", "pubchem_cid"] + PROV,
    "publications": ["pmid", "title", "year", "journal"] + PROV,
    "go_annotations": ["protein_id", "go_id", "go_name", "aspect"] + PROV,
}

def tidy(df):
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].map(lambda v: (v.strip() or None) if isinstance(v, str) else v)
    return df

def frame(rows, cols, extra=()):
    return tidy(pd.DataFrame(rows, columns=list(cols) + list(extra)))

def clean_genes(rows):
    df = frame(rows, COLS["genes"])
    df["gene_symbol"] = df["gene_symbol"].str.upper()
    return df.dropna(subset=["gene_symbol"]).drop_duplicates("gene_symbol").sort_values("gene_symbol")

def clean_proteins(rows):
    df = frame(rows, COLS["proteins"], extra=["uniprot_geneid"])
    df["gene_symbol"] = df["gene_symbol"].str.upper()
    return df.dropna(subset=["protein_id"]).drop_duplicates("protein_id").sort_values("protein_id")

def clean_hormones(rows):
    df = frame(rows, COLS["hormones"])
    return df.dropna(subset=["hormone_name"]).drop_duplicates("hormone_name").sort_values("hormone_name")

def clean_biomarkers(rows):
    df = frame(rows, COLS["biomarkers"])
    return df.dropna(subset=["biomarker_name"]).drop_duplicates("biomarker_name").sort_values("biomarker_name")

def clean_pathways(rows):
    df = frame(rows, COLS["pathways"])
    return (df.dropna(subset=["pathway_name", "reactome_id"])
              .drop_duplicates("reactome_id").drop_duplicates("pathway_name").sort_values("reactome_id"))

def clean_therapeutics(rows):
    df = frame(rows, COLS["therapeutics"])
    return df.dropna(subset=["drug_name"]).drop_duplicates("drug_name").sort_values("drug_name")

def clean_publications(rows):
    df = frame(rows, COLS["publications"])
    df["pmid"] = df["pmid"].astype("string").str.strip()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    return df.dropna(subset=["pmid"]).drop_duplicates("pmid").sort_values("pmid")

def clean_go(rows):
    df = frame(rows, COLS["go_annotations"])
    return df.dropna(subset=["protein_id", "go_id"]).drop_duplicates(["protein_id", "go_id"]).sort_values(["protein_id", "go_id"])

clean_df = {
    "genes": clean_genes(raw["genes"]),
    "proteins": clean_proteins(raw["proteins"]),
    "hormones": clean_hormones(raw["hormones"]),
    "biomarkers": clean_biomarkers(raw["biomarkers"]),
    "pathways": clean_pathways(raw["pathways"]),
    "therapeutics": clean_therapeutics(raw["therapeutics"]),
    "publications": clean_publications(raw["publications"]),
    "go_annotations": clean_go(raw["go"]),
}
# relationship material for Person B (not ingested here; B owns the junction tables)
tidy(pd.DataFrame(raw["protein_pathway"], columns=["protein_id", "gene_symbol", "reactome_id"])).drop_duplicates() \
    .to_csv(CLEAN / "protein_pathway_map.csv", index=False)
tidy(pd.DataFrame(raw["entity_pubs"], columns=["entity_type", "entity_name", "pmid"])).drop_duplicates() \
    .to_csv(CLEAN / "entity_publication_map.csv", index=False)

for name, df in clean_df.items():
    df.to_csv(CLEAN / f"{name}.csv", index=False)
    print(f"{name:15s} {len(df):5d} rows")

genes              36 rows
proteins           36 rows
hormones           17 rows
biomarkers          7 rows
pathways          144 rows
therapeutics       13 rows
publications      173 rows
go_annotations   1069 rows


## Step 4: Identifier mapping

Builds the master cross-reference `gene_symbol | ncbi_gene_id | uniprot_accession`. UniProt's own `GeneID` cross-reference is an independent check on the NCBI ID: a mismatch is logged and the UniProt accession is **not** attached to that gene.

Canonical keys: gene symbol (genes), UniProt accession (proteins), Reactome stable ID (pathways), PubChem CID (drugs, and hormones where one exists), PMID (publications).

In [8]:
genes, proteins = clean_df["genes"].copy(), clean_df["proteins"].copy()

xref = genes[["gene_symbol", "ncbi_gene_id"]].merge(
    proteins[["gene_symbol", "protein_id", "uniprot_geneid"]], on="gene_symbol", how="left")
xref = xref.rename(columns={"protein_id": "uniprot_accession"})

def status(r):
    if pd.isna(r.uniprot_accession):
        return "no_uniprot"
    if pd.isna(r.uniprot_geneid):
        return "unverified"          # UniProt entry has no GeneID cross-reference
    return "match" if str(r.uniprot_geneid) == str(r.ncbi_gene_id) else "MISMATCH"

xref["check"] = xref.apply(status, axis=1)
for _, r in xref[xref.check.isin(["MISMATCH", "unverified", "no_uniprot"])].iterrows():
    log_unresolved("gene", r.gene_symbol, f"cross-reference check = {r.check} "
                   f"(NCBI {r.ncbi_gene_id}, UniProt GeneID {r.uniprot_geneid})")

attach = xref[xref.check.isin(["match", "unverified"])].set_index("gene_symbol")["uniprot_accession"]
genes["uniprot_accession"] = genes["gene_symbol"].map(attach)
clean_df["genes"] = genes
genes.to_csv(CLEAN / "genes.csv", index=False)
xref.to_csv(CLEAN / "gene_xref.csv", index=False)

print(xref["check"].value_counts().to_string(), "\n")
xref

check
match    36 



,gene_symbol,ncbi_gene_id,uniprot_accession,uniprot_geneid,check
0,AMH,268,P03971,268,match
1,AMHR2,269,Q16671,269,match
2,AOPEP,84909,Q8N6M6,84909,match
3,AR,367,P10275,367,match
4,CHEK2,11200,O96017,11200,match
5,CYP11A1,1583,P05108,1583,match
6,CYP17A1,1586,P05093,1586,match
7,CYP19A1,1588,P11511,1588,match
8,DENND1A,57706,Q8TEH3,57706,match
9,ERBB4,2066,Q15303,2066,match


## Step 5: Revised schema

B's key names are untouched. Additions: external ID columns, provenance columns on every table (`source`, `source_accession`, `retrieved_on`), a gene link on `proteins`, a `reactome_id` on `pathways`, and a `go_annotations` table that feeds Hormone → Biological Process.

SQLite lets a TEXT primary key be NULL, so Step 7 checks that explicitly.

In [9]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS genes (
  gene_symbol TEXT PRIMARY KEY, name TEXT, ncbi_gene_id TEXT, uniprot_accession TEXT, description TEXT,
  source TEXT, source_accession TEXT, retrieved_on TEXT);

CREATE TABLE IF NOT EXISTS proteins (
  protein_id TEXT PRIMARY KEY, name TEXT, function TEXT, gene_symbol TEXT,
  source TEXT, source_accession TEXT, retrieved_on TEXT,
  FOREIGN KEY(gene_symbol) REFERENCES genes(gene_symbol));

CREATE TABLE IF NOT EXISTS hormones (
  hormone_name TEXT PRIMARY KEY, biological_role TEXT, pubchem_cid TEXT, uniprot_accession TEXT,
  source TEXT, source_accession TEXT, retrieved_on TEXT);

CREATE TABLE IF NOT EXISTS biomarkers (
  biomarker_name TEXT PRIMARY KEY, evidence TEXT, description TEXT, uniprot_accession TEXT,
  source TEXT, source_accession TEXT, retrieved_on TEXT);

CREATE TABLE IF NOT EXISTS pathways (
  pathway_name TEXT PRIMARY KEY, reactome_id TEXT UNIQUE, description TEXT,
  source TEXT, source_accession TEXT, retrieved_on TEXT);

CREATE TABLE IF NOT EXISTS therapeutics (
  drug_name TEXT PRIMARY KEY, mechanism TEXT, pubchem_cid TEXT,
  source TEXT, source_accession TEXT, retrieved_on TEXT);

CREATE TABLE IF NOT EXISTS publications (
  pmid TEXT PRIMARY KEY, title TEXT, year INTEGER, journal TEXT,
  source TEXT, source_accession TEXT, retrieved_on TEXT);

CREATE TABLE IF NOT EXISTS go_annotations (
  protein_id TEXT, go_id TEXT, go_name TEXT, aspect TEXT,
  source TEXT, source_accession TEXT, retrieved_on TEXT,
  PRIMARY KEY (protein_id, go_id),
  FOREIGN KEY(protein_id) REFERENCES proteins(protein_id));
"""

def connect():
    """Every connection in every notebook must go through this (or run the same PRAGMA)."""
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

if REBUILD_DB and DB_PATH.exists():
    DB_PATH.unlink()
conn = connect()
conn.executescript(SCHEMA)
conn.execute("PRAGMA foreign_keys = ON;")   # executescript can reset the session; set again
conn.commit()
print("Tables:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")])
print("foreign_keys =", conn.execute("PRAGMA foreign_keys").fetchone()[0])

Tables: ['biomarkers', 'genes', 'go_annotations', 'hormones', 'pathways', 'proteins', 'publications', 'therapeutics']
foreign_keys = 1


## Step 6: Idempotent ingestion

Parameterised `executemany` with upserts (`ON CONFLICT ... DO UPDATE`), so a corrected record replaces the old one instead of being silently skipped. Tables load in dependency order. Re-running leaves the database in the same state.

In [10]:
def upsert(conn, table, df, pk):
    cols = [r[1] for r in conn.execute(f"PRAGMA table_info({table})")]
    pkcols = [pk] if isinstance(pk, str) else list(pk)
    df = df.reindex(columns=cols)
    rows = [tuple(None if pd.isna(v) else (v.item() if hasattr(v, "item") else v) for v in r)
            for r in df.itertuples(index=False, name=None)]
    updates = ", ".join(f"{c}=excluded.{c}" for c in cols if c not in pkcols)
    sql = (f"INSERT INTO {table} ({', '.join(cols)}) VALUES ({', '.join('?' * len(cols))}) "
           f"ON CONFLICT({', '.join(pkcols)}) DO UPDATE SET {updates}")
    conn.executemany(sql, rows)
    return len(rows)

INGEST_ORDER = [("genes", "gene_symbol"), ("proteins", "protein_id"), ("pathways", "pathway_name"),
                ("hormones", "hormone_name"), ("biomarkers", "biomarker_name"),
                ("therapeutics", "drug_name"), ("publications", "pmid"),
                ("go_annotations", ("protein_id", "go_id"))]

conn = connect()
with conn:   # one transaction: all tables load or none do
    for table, pk in INGEST_ORDER:
        n = upsert(conn, table, clean_df[table], pk)
        print(f"{table:15s} upserted {n}")
print("Ingestion complete.")

genes           upserted 36
proteins        upserted 36
pathways        upserted 144
hormones        upserted 17
biomarkers      upserted 7
therapeutics    upserted 13
publications    upserted 173
go_annotations  upserted 1069
Ingestion complete.


In [11]:
# Idempotency test: a second run must not change row counts or content
before = {t: conn.execute(f"SELECT * FROM {t} ORDER BY 1,2").fetchall() for t, _ in INGEST_ORDER}
with conn:
    for table, pk in INGEST_ORDER:
        upsert(conn, table, clean_df[table], pk)
after = {t: conn.execute(f"SELECT * FROM {t} ORDER BY 1,2").fetchall() for t, _ in INGEST_ORDER}
assert before == after, "ingestion is not idempotent"
print("Idempotent: second ingestion left the database unchanged.")

Idempotent: second ingestion left the database unchanged.


## Step 7: Validation

Assertions are collected and reported together, then the cell raises if anything failed. The last block prints random records for the manual spot-check, which cannot be automated.

In [12]:
failures = []
def check(name, ok, detail=""):
    print(("PASS  " if ok else "FAIL  ") + name + ("" if ok else f"  -> {detail}"))
    if not ok:
        failures.append(name)

conn = connect()
q = lambda sql, *a: conn.execute(sql, a).fetchall()
KEYS = {"genes": "gene_symbol", "proteins": "protein_id", "hormones": "hormone_name",
        "biomarkers": "biomarker_name", "pathways": "pathway_name", "therapeutics": "drug_name",
        "publications": "pmid"}

# 1. no NULL primary keys, no duplicates after case normalisation
for t, k in KEYS.items():
    check(f"{t}.{k} has no NULLs", q(f"SELECT COUNT(*) FROM {t} WHERE {k} IS NULL OR TRIM({k})=''")[0][0] == 0)
    dups = q(f"SELECT LOWER({k}), COUNT(*) FROM {t} GROUP BY LOWER({k}) HAVING COUNT(*)>1")
    check(f"{t}.{k} has no case-insensitive duplicates", not dups, dups)

# 2. format checks
UNIPROT = re.compile(r"^([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2})$")
FORMATS = [("publications", "pmid", re.compile(r"^\d+$")),
           ("genes", "ncbi_gene_id", re.compile(r"^\d+$")),
           ("genes", "uniprot_accession", UNIPROT), ("proteins", "protein_id", UNIPROT),
           ("hormones", "uniprot_accession", UNIPROT), ("biomarkers", "uniprot_accession", UNIPROT),
           ("pathways", "reactome_id", re.compile(r"^R-HSA-\d+$")),
           ("therapeutics", "pubchem_cid", re.compile(r"^\d+$")), ("hormones", "pubchem_cid", re.compile(r"^\d+$")),
           ("go_annotations", "go_id", re.compile(r"^GO:\d{7}$"))]
for t, col, rx in FORMATS:
    bad = [v for (v,) in q(f"SELECT {col} FROM {t} WHERE {col} IS NOT NULL") if not rx.match(str(v))]
    check(f"{t}.{col} matches {rx.pattern[:28]}", not bad, bad[:5])

# 3. provenance on every row of every table
for t, _ in INGEST_ORDER:
    n = q(f"SELECT COUNT(*) FROM {t} WHERE source IS NULL OR retrieved_on IS NULL OR source_accession IS NULL")[0][0]
    check(f"{t} rows all have source, source_accession, retrieved_on", n == 0, f"{n} rows")

# 4. foreign keys
check("PRAGMA foreign_key_check is empty", q("PRAGMA foreign_key_check") == [], q("PRAGMA foreign_key_check")[:5])

# 5. every gene has a UniProt accession, or is explicitly logged
no_up = {g for (g,) in q("SELECT gene_symbol FROM genes WHERE uniprot_accession IS NULL")}
logged = {u["name"].upper() for u in unresolved if u["entity_type"] == "gene"}
check("every gene has UniProt or is logged as unresolved", no_up <= logged, sorted(no_up - logged))

# 6. known values from the progress document
check("CYP17A1 = Gene ID 1586", q("SELECT ncbi_gene_id FROM genes WHERE gene_symbol='CYP17A1'") == [("1586",)])
check("CYP17A1 = UniProt P05093", q("SELECT uniprot_accession FROM genes WHERE gene_symbol='CYP17A1'") == [("P05093",)])
check("Metformin = CID 4091", q("SELECT pubchem_cid FROM therapeutics WHERE drug_name='Metformin'") == [("4091",)])

# 7. B's contract: key columns still exist
for t, k in [("genes", "gene_symbol"), ("proteins", "protein_id"), ("pathways", "pathway_name"),
             ("therapeutics", "drug_name"), ("publications", "pmid")]:
    check(f"contract column {t}.{k} exists", k in [r[1] for r in q(f"PRAGMA table_info({t})")])

# 8. row counts
counts = pd.DataFrame([(t, q(f"SELECT COUNT(*) FROM {t}")[0][0]) for t, _ in INGEST_ORDER], columns=["table", "rows"])
print("\nRow counts:\n", counts.to_string(index=False))
print(f"\n{len(unresolved)} unresolved items logged")
assert not failures, f"{len(failures)} validation check(s) failed: {failures}"
print("\nAll validation checks passed.")

PASS  genes.gene_symbol has no NULLs
PASS  genes.gene_symbol has no case-insensitive duplicates
PASS  proteins.protein_id has no NULLs
PASS  proteins.protein_id has no case-insensitive duplicates
PASS  hormones.hormone_name has no NULLs
PASS  hormones.hormone_name has no case-insensitive duplicates
PASS  biomarkers.biomarker_name has no NULLs
PASS  biomarkers.biomarker_name has no case-insensitive duplicates
PASS  pathways.pathway_name has no NULLs
PASS  pathways.pathway_name has no case-insensitive duplicates
PASS  therapeutics.drug_name has no NULLs
PASS  therapeutics.drug_name has no case-insensitive duplicates
PASS  publications.pmid has no NULLs
PASS  publications.pmid has no case-insensitive duplicates
PASS  publications.pmid matches ^\d+$
PASS  genes.ncbi_gene_id matches ^\d+$
PASS  genes.uniprot_accession matches ^([OPQ][0-9][A-Z0-9]{3}[0-9]
PASS  proteins.protein_id matches ^([OPQ][0-9][A-Z0-9]{3}[0-9]
PASS  hormones.uniprot_accession matches ^([OPQ][0-9][A-Z0-9]{3}[0-9]
PASS 

In [13]:
# Manual spot-check: open each record on the live site and compare (5-10 records)
random.seed(7)
for t, k in [("genes", "gene_symbol"), ("proteins", "protein_id"), ("therapeutics", "drug_name"),
             ("pathways", "reactome_id"), ("publications", "pmid")]:
    print(f"--- {t}")
    for r in random.sample(q(f"SELECT * FROM {t}"), min(2, q(f"SELECT COUNT(*) FROM {t}")[0][0])):
        print("   ", tuple(str(x)[:70] if x is not None else None for x in r))

--- genes
    ('IRF1', 'interferon regulatory factor 1', '3659', 'P10914', 'The protein encoded by this gene is a transcriptional regulator and tu', 'NCBI Gene', '3659', '2026-10-12')
    ('ERBB4', 'erb-b2 receptor tyrosine kinase 4', '2066', 'Q15303', 'This gene is a member of the Tyr protein kinase family and the epiderm', 'NCBI Gene', '2066', '2026-10-12')
--- proteins
    ('P63096', 'Guanine nucleotide-binding protein G(i) subunit alpha-1', 'Guanine nucleotide-binding proteins (G proteins) function as transduce', 'GNAI1', 'UniProt', 'P63096', '2026-10-12')
    ('P03372', 'Estrogen receptor', 'Nuclear hormone receptor. The steroid hormones and their receptors are', 'ESR1', 'UniProt', 'P03372', '2026-10-12')
--- therapeutics
    ('Clomiphene', 'Clomifene has both estrogenic and anti-estrogenic properties, but its ', '2800', 'PubChem', '2800', '2026-10-12')
    ('Metformin', "Metformin's mechanisms of action are unique from other classes of oral", '4091', 'PubChem', '4091', '2026-10-1

In [14]:
# Handoff files for B and C: unresolved list and data dictionary (appended to README.md)
pd.DataFrame(unresolved, columns=["entity_type", "name", "reason"]).to_csv("unresolved_mappings.csv", index=False)

MEANING = {"source": "database the record came from", "source_accession": "ID of the record in that database",
           "retrieved_on": "UTC date the raw response was fetched (see data/raw/)",
           "ncbi_gene_id": "NCBI Gene ID", "uniprot_accession": "UniProt accession (reviewed, human)",
           "reactome_id": "Reactome stable ID", "pubchem_cid": "PubChem compound ID", "pmid": "PubMed ID",
           "go_id": "Gene Ontology term ID"}
lines = ["", "| table | column | type | key | meaning |", "|---|---|---|---|---|"]
for t, _ in INGEST_ORDER:
    for _, c, typ, _, _, pk in q(f"PRAGMA table_info({t})"):
        lines.append(f"| {t} | {c} | {typ} | {'PK' if pk else ''} | {MEANING.get(c, '')} |")
readme = Path("README.md").read_text(encoding="utf-8").split("Generated at the end of Step 7")[0]
Path("README.md").write_text(readme + "Generated by the notebook.\n" + "\n".join(lines) + "\n", encoding="utf-8")
print("Wrote unresolved_mappings.csv and the data dictionary in README.md")

Wrote unresolved_mappings.csv and the data dictionary in README.md
